In [ ]:
# ============================================
# NOTEBOOK 2 FIXED: Bone Diagram + MediaPipe Processing
# ============================================

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Cài đặt MediaPipe
!pip uninstall -y mediapipe
!pip install -q mediapipe==0.10.14

import mediapipe as mp

# Cấu hình
IMG_SIZE = 224  # Giảm xuống 224 để train nhanh hơn
INPUT_DATA_DIR = "/kaggle/input/datasets/tonirighthere/processed-data"
OUTPUT_DIR = "/kaggle/working/bone_data"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/bone_images", exist_ok=True)

print("=" * 50)
print("GIAI ĐOẠN 2: BONE DIAGRAM + MEDIAPIPE (FIXED)")
print("=" * 50)

# Khởi tạo MediaPipe Hands
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,  # Chỉ lấy 1 tay cho đơn giản
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# Định nghĩa các kết nối xương
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),  # Ngón cái
    (0, 5), (5, 6), (6, 7), (7, 8),  # Ngón trỏ
    (0, 9), (9, 10), (10, 11), (11, 12),  # Ngón giữa
    (0, 13), (13, 14), (14, 15), (15, 16),  # Ngón áp út
    (0, 17), (17, 18), (18, 19), (19, 20),  # Ngón út
]

def create_bone_diagram(keypoints, img_size=224):
    """Tạo bone diagram từ keypoints"""
    img = np.zeros((img_size, img_size, 3), dtype=np.uint8)  # Nền đen tốt hơn
    
    if len(keypoints) == 0:
        return img
    
    h, w = img_size, img_size
    scaled_pts = []
    
    # Lọc keypoints hợp lệ (x,y trong khoảng 0-1)
    for pt in keypoints:
        x, y = pt[0], pt[1]
        if 0 <= x <= 1 and 0 <= y <= 1:
            scaled_pts.append((int(x * w), int(y * h)))
        else:
            scaled_pts.append((0, 0))
    
    # Vẽ các kết nối (xương) - màu trắng trên nền đen
    for connection in HAND_CONNECTIONS:
        if connection[0] < len(scaled_pts) and connection[1] < len(scaled_pts):
            pt1 = scaled_pts[connection[0]]
            pt2 = scaled_pts[connection[1]]
            if pt1 != (0, 0) and pt2 != (0, 0):
                cv2.line(img, pt1, pt2, (255, 255, 255), 2)
    
    # Vẽ các keypoints
    for pt in scaled_pts:
        if pt != (0, 0):
            cv2.circle(img, pt, 3, (0, 255, 0), -1)
    
    return img

def extract_keypoints_and_bone(image_path):
    """Trích xuất keypoints và tạo bone diagram"""
    img = cv2.imread(image_path)
    if img is None:
        return None, None
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)
    
    if not results.multi_hand_landmarks:
        return None, None
    
    # Lấy bàn tay đầu tiên
    hand_landmarks = results.multi_hand_landmarks[0]
    
    # Trích xuất 21 keypoints
    keypoints = []
    for lm in hand_landmarks.landmark:
        keypoints.append([lm.x, lm.y, lm.z])
    
    # Tạo bone diagram
    bone_img = create_bone_diagram(keypoints, IMG_SIZE)
    
    return bone_img, keypoints

# Đọc metadata
print("\n1. Đọc metadata...")
with open(f"{INPUT_DATA_DIR}/metadata.json", 'r') as f:
    metadata = json.load(f)

classes = metadata['classes']
class_mapping = metadata['class_mapping']  # {'A': 0, 'B': 1, ...}
print(f"Classes: {len(classes)}")
print(f"Class mapping: {dict(list(class_mapping.items())[:5])}")

# Tìm đúng đường dẫn đến ảnh
print("\n2. Tìm đường dẫn ảnh...")

# Thử các đường dẫn khác nhau
possible_paths = [
    f"{INPUT_DATA_DIR}/asl_processed/images",
    f"{INPUT_DATA_DIR}/images",
    f"{INPUT_DATA_DIR}/processed_data/images",
    "/kaggle/input/processed-data/images",
]

images_base_dir = None
for path in possible_paths:
    if os.path.exists(path):
        images_base_dir = path
        print(f"Found images at: {path}")
        break

if images_base_dir is None:
    # Liệt kê để debug
    print("Listing directory contents:")
    !ls -la {INPUT_DATA_DIR}
    raise Exception("Không tìm thấy thư mục images!")

# Xử lý dữ liệu
print("\n3. Đang tạo bone diagram...")

all_keypoints = []
bone_info = []
failed_classes = []

for class_name in tqdm(classes, desc="Processing classes"):
    class_dir = f"{images_base_dir}/{class_name}"
    
    if not os.path.exists(class_dir):
        failed_classes.append(class_name)
        continue
    
    output_class_dir = f"{OUTPUT_DIR}/bone_images/{class_name}"
    os.makedirs(output_class_dir, exist_ok=True)
    
    images = [f for f in os.listdir(class_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    
    # THAY ĐỔI Ở ĐÂY:
    # Nếu là M hoặc N thì lấy tối đa (ví dụ 800 ảnh), các class khác lấy 300
    if class_name in ['M', 'N']:
        max_samples = 800  # Tăng quota để bù đắp tỉ lệ fail cao
    else:
        max_samples = 300

    success_count = 0
    for img_name in images:
        if success_count >= max_samples: # Đủ chỉ tiêu thì dừng
            break
        img_path = f"{class_dir}/{img_name}"
        
        bone_img, keypoints = extract_keypoints_and_bone(img_path)
        
        if bone_img is not None and keypoints is not None:
            # Lưu bone diagram
            bone_path = f"{output_class_dir}/{img_name.replace('.jpg', '.png')}"
            cv2.imwrite(bone_path, bone_img)
            
            # Lưu keypoints
            all_keypoints.append({
                'class': class_name,
                'image': img_name,
                'keypoints': keypoints
            })
            
            bone_info.append({
                'class': class_name,
                'image': img_name,
                'bone_path': bone_path
            })
            success_count += 1
    
    if success_count == 0:
        failed_classes.append(class_name)
    else:
        print(f"  {class_name}: {success_count} bone diagrams")

print(f"\n✅ Đã tạo {len(bone_info)} bone diagrams")
if failed_classes:
    print(f"⚠️ Classes failed: {failed_classes}")
